
# T3.3 EIT-003 Lifecycle Environmental Performance Model

READ.ME 

This notebook is the **shared foundation** for the EIT-003 pipeline. It contains everything that is independent of runtime event data and must be run once before EIT-001 or EIT-002 can be used.

- Environment setup (Brightway project initialisation ecoinvent and BONSAI database installation)
- Foreground model: REED foreground database imported from `REED_model_*.xlsx`
- Baseline LCA results 
  - KOMATSU results
  - WWR results
  - GOIMEK A1 / A2 / A3 results
- Analysis & visualisation (Contribution analysis, contribution plots)
- Reusable LCA functions (`lca_score`, `run_contribution_analysis`, `lcia_for_activity_with_scenario`)

Run this notebook **once** to set up the Brightway project.

> EIT-001 and EIT-002 both assume this project is initialised and call into it.

### 1. Importers

#### 1.1 Brightway

In [26]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import Config
import bw_processing as bwp
from bw2io.strategies.generic import set_code_by_activity_hash


import matplotlib.pyplot as plt
import d3graph
import polyviz

from datetime import date

In [27]:
config = Config()

In [28]:
bd.projects.set_current(config.PROJECT_NAME)

In [29]:
bi.remote.install_project(project_key="ecoinvent-3.10-biosphere", overwrite_existing=True)

Restoring project backup archive - this could take a few minutes...
Restored project: ecoinvent-3.10-biosphere


'ecoinvent-3.10-biosphere'

#### 1.2 BONSAI and REED database

In [ ]:
import import_db

In [31]:
bonsai = bd.Database('BONSAI_V2.1.3')
bonsai_bio = bd.Database('BONSAI_V2.1.3 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')

In [ ]:
del bd.databases["your_database_name"]

In [32]:
bonsai.search('chemicals')

['chemicals nec(market for)' (ton, RU, None),
 'chemicals nec(market for)' (ton, GR, None),
 'chemicals nec(market for)' (ton, WE, None),
 'chemicals nec(market for)' (ton, WM, None),
 'chemicals nec(market for)' (ton, HR, None),
 'chemicals nec(market for)' (ton, MX, None),
 'chemicals nec(market for)' (ton, BE, None),
 'chemicals nec(market for)' (ton, LT, None),
 'chemicals nec(market for)' (ton, CN, None),
 'chemicals nec(market for)' (ton, CH, None),
 'chemicals nec(market for)' (ton, SE, None),
 'chemicals nec(market for)' (ton, WA, None),
 'chemicals nec(market for)' (ton, NO, None),
 'chemicals nec(market for)' (ton, DE, None),
 'chemicals nec(market for)' (ton, DK, None),
 'chemicals nec(market for)' (ton, EE, None),
 'chemicals nec(market for)' (ton, CY, None),
 'chemicals nec(market for)' (ton, LV, None),
 'chemicals nec(market for)' (ton, US, None),
 'chemicals nec(market for)' (ton, PT, None),
 'chemicals nec(market for)' (ton, JP, None),
 'chemicals nec(market for)' (ton,

In [33]:
bd.databases

Databases dictionary with 4 object(s):
	BONSAI_V2.1.3
	BONSAI_V2.1.3 biosphere
	ecoinvent-3.10-biosphere
	reed

In [34]:
from bw2io.importers import ExcelImporter
from reed.utils import Config

importer  = ExcelImporter(config.REED_PATH/'model'/'REED_model_20260407_no_biosphere.xlsx',)
importer.apply_strategies()
importer.match_database(fields=['name','unit','location'])
importer.match_database(db_name=bonsai.name,fields=['name','unit','location','activity type'])
importer.match_database(db_name=bd.config.biosphere,fields=['name','categories','unit'])
reed = bd.Database('reed')

Extracted 6 worksheets in 1.56 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 3.58 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


In [35]:
importer.statistics()

Graph statistics for `reed` importer:
77 graph nodes:
	None: 77
423 graph edges:
	technosphere: 344
	production: 77
	biosphere: 2
423 edges to the following databases:
	BONSAI_V2.1.3: 240
	reed: 181
	ecoinvent-3.10-biosphere: 2
0 unique unlinked edges (0 total):




(77, 423, 0, 0)

In [36]:
importer.write_database()

100%|██████████| 77/77 [00:00<00:00, 301.57it/s]


09:33:50+0200 [info     ] Vacuuming database            
Created database: reed


In [37]:
pd.DataFrame(importer.unlinked)

""


In [38]:
ecoinvent.search('aluminium',limit=None)

['Aluminium III' (kilogram, None, ('air',)),
 'Aluminium III' (kilogram, None, ('water',)),
 'Aluminium III' (kilogram, None, ('soil',)),
 'Aluminium III' (kilogram, None, ('water', 'ground-')),
 'Aluminium III' (kilogram, None, ('soil', 'industrial')),
 'Aluminium III' (kilogram, None, ('water', 'ocean')),
 'Aluminium III' (kilogram, None, ('soil', 'forestry')),
 'Aluminium III' (kilogram, None, ('soil', 'agricultural')),
 'Aluminium III' (kilogram, None, ('water', 'surface water')),
 'Aluminium III' (kilogram, None, ('water', 'ground-, long-term')),
 'Aluminium III' (kilogram, None, ('air', 'low population density, long-term')),
 'Aluminium III' (kilogram, None, ('air', 'urban air close to ground')),
 'Aluminium III' (kilogram, None, ('air', 'non-urban air or from high stacks')),
 'Aluminium hydroxide' (kilogram, None, ('water',)),
 'Aluminium' (kilogram, None, ('natural resource', 'in ground'))]

In [39]:
data_reed = [activity.as_dict() for activity in reed]
df_reed = pd.DataFrame(data_reed)


In [40]:
from IPython.display import display
display(df_reed[['name', 'comment', 'location', 'unit', 'database']])

,name,comment,location,unit,database
0,"drilling, soraluce",proxy for drilling. There are multiple differe...,DE,hour,reed
1,"Soraluce, average",This inventory is calculated based on informat...,ES,unit,reed
2,KOMATSU total,None,GB,unit,reed
3,"welding, manual","draft for manual welding process (MIG), missin...",DE,hour,reed
4,"Soraluce machining, GOIMEK",None,ES,unit,reed
...,...,...,...,...,...
72,KOMATSU A1: raw materials,None,GB,unit,reed
73,"milling, soraluce",proxy for milling. Waiting for primary electri...,GB,hour,reed
74,WWR total,None,DE,unit,reed
75,"Soraluce, average",This inventory is calculated based on informat...,DE,unit,reed


In [41]:
bonsai = bd.Database('BONSAI_V2.1.3')
bonsai_bio = bd.Database('BONSAI_V2.1.3 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')
reed = bd.Database('reed')

In [42]:
bd.databases

Databases dictionary with 4 object(s):
	BONSAI_V2.1.3
	BONSAI_V2.1.3 biosphere
	ecoinvent-3.10-biosphere
	reed

# Supplementary Material

### 2. Results without Footprint Monitoring

In [ ]:
reed.search('goimek')

In [ ]:
komatsu_processes = {
    "KOMATSU total": reed.get(name='KOMATSU total', location="GB", unit='unit'),
    "KOMATSU A1: raw materials": reed.get(name='KOMATSU A1: raw materials', location="GB", unit='unit'),
    "KOMATSU A2: transport": reed.get(name='KOMATSU A2: transport', location="GB", unit='unit'),
    "KOMATSU A3: machining": reed.get(name='KOMATSU A3: machining', location="GB", unit='unit'),
    
}

In [ ]:
wwr_processes = {
    "WWR total": reed.get(name='WWR total', location="DE", unit='unit'),
    "WWR A1: material inputs": reed.get(name='WWR A1: material inputs', location="DE", unit='unit'),
    "WWR A2: transport": reed.get(name='WWR A2: transport', location="DE", unit='unit'),
    "WWR A3: machining": reed.get(name='WWR A3: machining', location="DE", unit='unit'),
}

In [ ]:
goimek_processes = {
    "GOIMEK total": reed.get(name='GOIMEK total', location="ES", unit='unit'),
    "GOIMEK A1: material input": reed.get(name='GOIMEK A1: material input', location="ES", unit='unit'),
    "GOIMEK A2: transport": reed.get(name='GOIMEK A2: transport', location="ES", unit='unit'),
    "GOIMEK A3: machining": reed.get(name='GOIMEK A3: machining', location="ES", unit='unit'),
}

In [ ]:
from reed.LCA_functions import (
    lca_score,
    run_contribution_analysis,
    plot_contribution
)

In [ ]:
df_komatsu_results = lca_score(komatsu_processes, config.lcia_gwp100)
display(df_komatsu_results)


In [ ]:
df_wwr_results = lca_score(wwr_processes, config.lcia_gwp100)
display(df_wwr_results)


In [ ]:
df_goimek_results = lca_score(goimek_processes, config.lcia_gwp100)
display(df_goimek_results)


In [ ]:
df_results_all = pd.concat(
    [df_komatsu_results, df_wwr_results, df_goimek_results],
    ignore_index=True
)
df_results_all

### 3. Plot Life Cycle Stages

In [ ]:
df_plot = df_komatsu_results.set_index("Process")

# Remove the row
df_plot = df_plot.drop("KOMATSU total", errors="ignore")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Life Cycle Phase in DPP")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
df_plot = df_wwr_results.set_index("Process")

# Remove the row
df_plot = df_plot.drop("WWR total", errors="ignore")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Life Cycle Phase in DPP")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
df_plot = df_goimek_results.set_index("Process")

# Remove the row
df_plot = df_plot.drop("GOIMEK total", errors="ignore")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Life Cycle Phase in DPP")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

### 4. Plot contribution analysis

In [ ]:
komatsu_ca = {}

for key, demand in komatsu_processes.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.001
    )
    komatsu_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )

In [ ]:
wwr_ca = {}

for key, demand in wwr_processes.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.001
    )
    wwr_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )

In [ ]:
goimek_ca = {}

for key, demand in goimek_processes.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.001
    )
    goimek_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )

In [ ]:
wwr_total = reed.get(name="WWR total", location="DE", unit="unit")

In [ ]:
bwa.print_recursive_calculation(wwr_total,
lcia_method=config.lcia_gwp100,max_level=1,cutoff=0.001)